In [2]:
import pandas as pd
import numpy as np

data = {
    'arquivo': [f'audio_{i}.wav' for i in range(100)] ,
    'locutor': [f'spk_{i % 10}' for i in range(100)],
    'label': ['bonafide' if i % 2 == 0 else 'spoof' for i in range(100)]
}
df_corpus = pd.DataFrame(data)
print(df_corpus)

         arquivo locutor     label
0    audio_0.wav   spk_0  bonafide
1    audio_1.wav   spk_1     spoof
2    audio_2.wav   spk_2  bonafide
3    audio_3.wav   spk_3     spoof
4    audio_4.wav   spk_4  bonafide
..           ...     ...       ...
95  audio_95.wav   spk_5     spoof
96  audio_96.wav   spk_6  bonafide
97  audio_97.wav   spk_7     spoof
98  audio_98.wav   spk_8  bonafide
99  audio_99.wav   spk_9     spoof

[100 rows x 3 columns]


In [5]:
locutores = df_corpus['locutor'].unique()
np.random.seed(42)
np.random.shuffle(locutores)

n_treino = int(0.6 * len(locutores))
n_dev = int(0.2 * len(locutores))

spk_train = locutores[:n_treino]
spk_dev = locutores[n_treino:n_treino+n_dev]
spk_test = locutores[n_treino+n_dev:]

def assign_partition(spk):
  if spk in spk_train: return 'train'
  elif spk in spk_dev: return 'dev'
  else: return 'test'

df_corpus['partition'] = df_corpus['locutor'].apply(assign_partition)
print(df_corpus['partition'].value_counts)

<bound method IndexOpsMixin.value_counts of 0     train
1     train
2     train
3      test
4       dev
      ...  
95    train
96     test
97    train
98    train
99      dev
Name: partition, Length: 100, dtype: object>


In [7]:
def check_leakage(df):
  train_spks = set(df[df['partition'] == 'train']['locutor'])
  dev_spks = set(df[df['partition'] == 'dev']['locutor'])
  test_spks = set(df[df['partition'] == 'test']['locutor'])

  leak_dev = train_spks.intersection(dev_spks)
  leak_test = train_spks.intersection(test_spks)

  if leak_dev or leak_test:
    raise ValueError("Locutor Leakage")
    print("No locutor leakage")

check_leakage(df_corpus)

In [10]:
def balance_classes(df, part='train'):
  df_part = df[df['partition'] == part]
  min_count = df_part['label'].value_counts().min()

  df_bonafide = df_part[df_part['label'] == 'bonafide'].sample(min_count, random_state=42)
  df_spoof = df_part[df_part['label'] == 'spoof'].sample(min_count, random_state = 42)

  return pd.concat([df_bonafide, df_spoof])

df_train_bal = balance_classes(df_corpus, 'train')

print(df_train_bal['label'].value_counts())

label
bonafide    30
spoof       30
Name: count, dtype: int64


In [11]:
df_corpus.to_csv('frozen_partition_v1.csv', index=False)

In [16]:
import pywt

print("Daubechies:", pywt.wavelist(family='db')[:10])
print("Symlets:", pywt.wavelist(family='sym')[:10])

sinal = np.random.randn(16000)
wp_db4 = pywt.WaveletPacket(data = sinal, wavelet = 'db4', mode='symmetric', maxlevel=5)

no_exemplo = wp_db4['aaaaa']
print(f"nome do no: {no_exemplo.path}")
print(f"tamanho dos coeficientes: {len(no_exemplo.data)}")

Daubechies: ['db1', 'db2', 'db3', 'db4', 'db5', 'db6', 'db7', 'db8', 'db9', 'db10']
Symlets: ['sym2', 'sym3', 'sym4', 'sym5', 'sym6', 'sym7', 'sym8', 'sym9', 'sym10', 'sym11']
nome do no: aaaaa
tamanho dos coeficientes: 506


In [19]:
wp_vazio = pywt.WaveletPacket(data=np.zeros_like(sinal), wavelet='db4', mode='symmetric')

wp_vazio['aaaaa'] = wp_db4['aaaaa'].data

sinal_filtred = wp_vazio.reconstruct(update=False)
print("Tamanho do sinal reconstruido:", len(sinal_filtred))

Tamanho do sinal reconstruido: 16000


In [21]:
def shannon_entropy(data):
  energia = data ** 2
  soma_energia = np.sum(energia)


  if soma_energia == 0:
    return 0

  p = energia / soma_energia
  p = p[p>0]

  return -np.sum(p*np.log(p))

entropia_pai = shannon_entropy(wp_db4['a'].data)

print(f"Entropia do pai 'a': {entropia_pai:.4f}")

Entropia do pai 'a': 8.2556


In [23]:
entropia_f1 = shannon_entropy(wp_db4['aa'].data)
entropia_f2 = shannon_entropy(wp_db4['ad'].data)
entropia_f_total = entropia_f1 + entropia_f2

if entropia_f_total < entropia_pai:
  print("dividir")
else:
  print("manter")

manter


In [27]:
wp_db8 = pywt.WaveletPacket(data=sinal, wavelet='db8', maxlevel=5)
wp_sym8 = pywt.WaveletPacket(data=sinal, wavelet='sym8', maxlevel=5)

print("DB8 energia total: ", np.sum(wp_db8['aaaaa'].data ** 2))
print("SYM8 Energia total: ", np.sum(wp_sym8['aaaaa'].data ** 2))

DB8 energia total:  713.5638403671619
SYM8 Energia total:  822.9324028653682


In [28]:
import librosa
import soundfile as sf
import os

sf.write('teste_dummy.wav', np.zeros(44100), 44100)

audio_lib, sr_lib = librosa.load('teste_dummy.wav', sr=None, mono=False)

audi_sf, sr_sf = sf.read('teste_dummy.wav')

In [29]:
def convert_mono(audio_array):
  if len(audio_array.shape) > 1:
    return np.mean(audio_array, axis = 1)
  return audio_array

def reamostrar(audio_array, sr_atual, sr_desejado = 16000):
  if sr_atual != sr_desejado:
    return librosa.resample(y=audio_array, orig_sr = sr_atual, target_sr = sr_desejado)
  return audio_array

def pipeline_audio(camino_arquivo):
  audio, sr = sf.read(camino_arquivo)

  audio = convert_mono(audio)

  audio = reamostrar(audio, sr, 16000)

  return audio, 16000

In [30]:
def info_audio(caminho):
  info = sf.info(caminho)
  return {
      'sr': info.samplerate,
      'canais' : info.channels,
      'duracao_s': info.duration,
      'amostras': info.frames
  }

print(info_audio('teste_dummy.wav'))

{'sr': 44100, 'canais': 1, 'duracao_s': 1.0, 'amostras': 44100}


In [32]:
import glob
from pathlib import Path

arquivos = glob.glob('meu_dataset/**/*.wav', recursive=True)
print(f"Quantidade de arquivos: {len(arquivos)}")

pasta = Path('.')

arquivos_wav = list(pasta.rglob('*.wav'))
for arq in arquivos_wav[:3]:
  print(arq.name, arq.stat().st_size, "bytes")

Quantidade de arquivos: 0
teste_dummy.wav 88244 bytes


In [33]:
def processar_lote(lista_caminhos, pasta_saida):
  os.makedirs(pasta_saida, exist_ok=True)

  for caminho in lista_caminhos:
    nome_arquivo = os.path.basename(caminho)
    caminho_saida = os.path.join(pasta_saida, nome_arquivo)

    aud, sr = pipeline_audio(caminho)

  sf.write(caminho_saida, aud, sr)